# Local RAG Knowledge Base with a Vector Database — Reference Notebook

> **Reference notebook.** See [`vector_databases.md`](./vector_databases.md) for the underlying
> embeddings/indexing/RAG concepts distilled here, and [`chroma.md`](../12-langchain-p1/chroma.md) for a
> Chroma-specific deep dive.

**Methods covered:**
- Loading a PDF into per-page documents (`PyPDFLoader`) and splitting them into overlapping chunks
  (`RecursiveCharacterTextSplitter`)
- Embedding chunks locally with a sentence-transformers model (`HuggingFaceEmbeddings`) and persisting
  them in a **Chroma** vector store
- Retrieving with **Maximal Marginal Relevance** (`max_marginal_relevance_search`) instead of plain
  similarity search, to reduce redundancy among the retrieved chunks
- Loading a local causal LM (`Qwen2.5-1.5B-Instruct`) and generating a grounded answer by hand: chat
  template -> tokenize -> `.generate()` -> decode

**Use this as a reference when:** you need a fully local RAG pipeline (local embeddings + local
generation, no external API), or want to see MMR retrieval instead of plain similarity search.

**Don't use this as a reference for:** API-backed RAG (Gemini/OpenAI embeddings + chat models — see
[`rag_and_chats.ipynb`](../05-prompt-engineering/rag_and_chats.ipynb)), multi-turn memory, or
agent-based retrieval.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoModelForCausalLM, AutoTokenizer

import warnings
warnings.filterwarnings("ignore")


In [ ]:
# PyPDFLoader produces one Document per PDF page -- page_content holds the page's text,
# metadata carries {"source": <path>, "page": <index>} alongside it.
pages = PyPDFLoader("./data/ArtigoDSA1.pdf").load()
len(pages)


In [ ]:
# RecursiveCharacterTextSplitter tries a hierarchy of separators (paragraph, then sentence, then
# word) to keep each chunk under chunk_size while still breaking on natural boundaries where possible.
# chunk_overlap repeats the last few characters of one chunk at the start of the next, so a fact
# sitting right on a chunk boundary isn't fully lost from either side.
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
docs = splitter.split_documents(pages)
len(docs)


In [ ]:
# HuggingFaceEmbeddings wraps a local sentence-transformers checkpoint -- embedding happens on this
# machine, no API call. The same model must be used again at query time: embeddings from two different
# models don't share a vector space, so similarity between them is meaningless.
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Chroma.from_documents embeds every chunk once and stores the vector alongside its source text and
# metadata. persist_directory writes the index to disk so it survives across notebook restarts instead
# of being rebuilt (and re-embedded) every run.
vectordb = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./chroma_store/",
)

# `_collection` is the underlying chromadb collection the langchain wrapper delegates to;
# count() confirms every chunk actually made it into the store.
vectordb._collection.count()


In [ ]:
# Plain similarity_search would just return the k closest vectors, which can cluster around the same
# passage if several chunks say near-identical things. max_marginal_relevance_search (MMR) instead
# fetches `fetch_k` candidates first, then greedily picks `k` of them by balancing relevance to the
# query against distance from chunks already chosen -- trading a little pure relevance for coverage.
question = "A pandemia do COVID-19 acelerou o ritmo do desenvolvimento digital em todo o mundo?"
relevant_chunks = vectordb.max_marginal_relevance_search(question, k=2, fetch_k=3)
relevant_chunks[0]


In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# AutoModelForCausalLM / AutoTokenizer resolve the correct model/tokenizer classes from the checkpoint's
# own config, so the same loading code works for any causal LM repo id, not just this one.
# torch_dtype="auto" and device_map="auto" let transformers pick the checkpoint's native precision and
# place weights on whatever device (GPU/CPU) is available, instead of hardcoding either.
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [ ]:
def ask(question, k=2, fetch_k=3, max_new_tokens=512):
    context = vectordb.max_marginal_relevance_search(question, k=k, fetch_k=fetch_k)

    prompt = f"""
Você é um assistente especialista. Você usa o contexto fornecido como sua base de conhecimento complementar para responder à pergunta.
context = {context}
question = {question}
answer =
"""
    messages = [
        {"role": "system", "content": "Você é Qwen, criado pela Alibaba Cloud. Você é um assistente especialista."},
        {"role": "user", "content": prompt},
    ]

    # apply_chat_template renders the {role, content} messages into the exact markup string
    # (special tokens included) the checkpoint was instruction-tuned to expect -- a raw
    # concatenation of the messages would not match what the model saw during fine-tuning.
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # generate() runs the autoregressive decoding loop. For a causal LM it returns the input token
    # ids followed by the newly generated ones concatenated together, so the input length is
    # sliced off below to keep only the continuation.
    generated_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens)
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    # skip_special_tokens=True drops the chat-template control tokens, leaving just the answer text.
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


In [ ]:
print(ask("A pandemia do COVID-19 acelerou o ritmo do desenvolvimento digital em todo o mundo?"))


In [ ]:
print(ask("Quantos empregos o Fórum Econômico Mundial estima que serão perdidos para automação nos próximos anos?"))


In [ ]:
print(ask("Qual é a habilidade mais importante na era da Inteligência Artificial?"))
